# Demo — Medallion Lineage & Blast-Radius Monitoring
In Veridian Health's Medallion Lakehouse, OpenLineage events track how **Silver** clinical and billing tables feed **Gold** regulatory and research products. Lineage answers two distinct operational questions:
1. **Forward blast radius (`trace_downstream`)**: If a Silver feed degrades or suffers a PHI breach, which Gold products are impacted?
2. **Backward provenance (`trace_upstream`)**: Which upstream Silver tables must be audited to verify a Gold metric or research cohort?

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your AWS Cloud Lab environment and seed the **Veridian Health Medallion Lakehouse** (`veridian_lakehouse` in AWS Glue, S3 `bronze/silver/gold` Parquet prefixes, and AWS Lake Formation enforcement) via `setup_aws.py`.

In [1]:
import os
os.environ["VERIDIAN_LF_LIVE"] = "1"
os.environ["VERIDIAN_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
glue = boto3.client("glue", region_name=os.environ["VERIDIAN_AWS_REGION"])
try:
    glue.get_database(Name="veridian_lakehouse")
    print("Glue database 'veridian_lakehouse' is already seeded and ready.")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

Glue database 'veridian_lakehouse' is already seeded and ready.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from demo_toolkit import connect_lakehouse, lh_backend

lh = connect_lakehouse()
print(f"Backend: {lh_backend()} -> {type(lh).__name__}")
display(lh.lineage.events())

Backend: live -> AwsMedallionLakehouse


                          job                                                inputs                            outputs
0       ingest_encounter_labs          silver_clinical_encounter, silver_lab_result          gold_encounter_turnaround
1     publish_cms_timely_care                             gold_encounter_turnaround               gold_cms_timely_care
2  build_patient_clinical_360           silver_patient_master, silver_claim_billing          gold_patient_clinical_360
3      export_research_cohort  gold_patient_clinical_360, silver_clinical_encounter  gold_deidentified_research_cohort


## 1. Trace downstream blast radius and upstream provenance

In [3]:
print("Downstream of silver_lab_result (CMS blast radius):", lh.lineage.trace_downstream("silver_lab_result"))
print("Upstream of gold_deidentified_research_cohort (HIPAA provenance):", lh.lineage.trace_upstream("gold_deidentified_research_cohort"))

Downstream of silver_lab_result (CMS blast radius): ['gold_encounter_turnaround', 'gold_cms_timely_care']
Upstream of gold_deidentified_research_cohort (HIPAA provenance): ['gold_patient_clinical_360', 'silver_clinical_encounter', 'silver_patient_master', 'silver_claim_billing']


## 2. Place a monitoring signal at the Silver -> Gold boundary

| Signal Type | Dataset / Boundary | Threshold | Owner | Regulatory Driver |
| :--- | :--- | :--- | :--- | :--- |
| **Unit-of-measure accuracy** | `gold_encounter_turnaround.turnaround_min` | `0 rows with turnaround_min > 240` | Diagnostics Lab Steward (`diagnostics-lab@veridian.health`) | CMS eCQM Timely Diagnostic Care accuracy |
| **Anomalous PHI access / breach** | `silver_patient_master` & `gold_patient_clinical_360` | `> 3x 7-day rolling read baseline or unauthorized role query` | Security & Chief Health Data Officer | HIPAA Breach Notification Rule (45 CFR §164.404) & GDPR Art. 33 (72h) |